# WF-E001/v001 — Validate the versioned notebook and result-evidence lifecycle

این نوت‌بوک فقط برای پروتکل ثبت‌شدهٔ همین نسخه است. هدف، فرضیه، ورودی‌ها،
معیارهای پذیرش و دستور دقیق در زیر ثبت شده‌اند. تغییر روش، seed، داده یا آستانه
نیاز به revision تازه در ریپو دارد. Run all اجرای همین آزمایش را آغاز می‌کند.

**مرحله:** workflow — **نوع:** infrastructure

**فرضیه:** A deterministic CPU fixture can be packaged, executed, imported and reviewed with exact provenance and all declared artifacts.

**ارجاع هدف:** research/decisions/ADR-0001.md; docs/OPERATIONAL_ROADMAP_FA.md workflow contract

```json
{
  "hypothesis": "A deterministic CPU fixture can be packaged, executed, imported and reviewed with exact provenance and all declared artifacts.",
  "goal_reference": "research/decisions/ADR-0001.md; docs/OPERATIONAL_ROADMAP_FA.md workflow contract",
  "parent": null,
  "command": [
    "{python}",
    "-B",
    "scripts/workflow_smoke.py"
  ],
  "inputs": [],
  "output_paths": [
    "runs/workflow_smoke"
  ],
  "required_artifacts": [
    "runs/workflow_smoke/metrics.json"
  ],
  "gates": [
    {
      "artifact": "runs/workflow_smoke/metrics.json",
      "name": "arithmetic",
      "op": "eq",
      "pointer": "/checks/arithmetic",
      "value": true
    },
    {
      "artifact": "runs/workflow_smoke/metrics.json",
      "name": "deterministic",
      "op": "eq",
      "pointer": "/checks/deterministic",
      "value": true
    },
    {
      "artifact": "runs/workflow_smoke/metrics.json",
      "name": "scope_is_infrastructure",
      "op": "eq",
      "pointer": "/scientific_evidence",
      "value": false
    }
  ],
  "resources": null,
  "compute": {
    "device": "cpu",
    "expected_seconds": 10,
    "gpu_required": false
  },
  "prerequisites": [
    "No pre-existing runs/workflow_smoke output directory"
  ],
  "dependencies": [
    "Python standard library only"
  ],
  "scientific_scope": "Infrastructure acceptance only. No ERGT model is trained, no M2 gate is passed, no M3 coupling is authorized.",
  "timeout_seconds": 60
}
```

Protocol SHA-256: `44fa6240e1bcbbf27195a8e3015b27b450fa65387b60c585803aced474eb4a27`

| معیار | فایل و مسیر مقدار | عملگر | مقدار لازم |
|---|---|---|---|
| arithmetic | runs/workflow_smoke/metrics.json/checks/arithmetic | eq | true |
| deterministic | runs/workflow_smoke/metrics.json/checks/deterministic | eq | true |
| scope_is_infrastructure | runs/workflow_smoke/metrics.json/scientific_evidence | eq | false |

این اجرا به‌تنهایی مجوز عبور فاز نیست؛ خروجی باید به ریپو برگردد، با قفل‌های
نسخه تطبیق داده شود، معیارها ارزیابی شوند و نتیجه‌گیری ثبت شود. شکست هم خروجی
پژوهشی است و باید نگه‌داری شود. کنترل داده و checkpoint با SHA-256 انجام می‌شود.

زمان و حافظه تابع تنظیمات پروتکل و runtime است؛ اگر در بخش resources یا compute مقدار ثبت
نشده، برآورد هنوز معلوم نیست. محیط اجرا و مدت واقعی خودکار در نتیجه ثبت می‌شوند.


## آماده‌سازی و دریافت نسخهٔ دقیق

ابتدا runtime مطابق بخش resources یا compute انتخاب شود. دو فایل ساخته‌شده در ریپو را باهم
بارگذاری کنید: ZIP سورس نسخه و `package.json` کنار همین نوت‌بوک. بسته عمداً
checkpointها، نتایج قبلی و محیط مجازی را ندارد. این سلول hash کل بسته و تک‌تک
فایل‌های آن را بررسی می‌کند و در پوشه‌ای تازه باز می‌کند.

هیچ سرویس محلی نمی‌تواند صرف ساخت این نوت‌بوک، اجرای Colab را ادعا کند. اجرای
واقعی با Run all در حساب پژوهشگر یا با ابزار مجاز و متصل انجام می‌شود.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, os, stat, sys, uuid, zipfile
from google.colab import files

EXPERIMENT_ID = 'WF-E001'
REVISION = 'v001'
PROTOCOL_SHA256 = '44fa6240e1bcbbf27195a8e3015b27b450fa65387b60c585803aced474eb4a27'
REVISION_RELATIVE = 'research/experiments/WF-E001/v001'
uploaded = files.upload()
if 'package.json' not in uploaded:
    raise RuntimeError('Upload package.json and its exact source ZIP together')
lock = json.loads(uploaded['package.json'])
assert lock['experiment_id'] == EXPERIMENT_ID and lock['revision'] == REVISION
assert lock['protocol_sha256'] == PROTOCOL_SHA256
archive_name = lock['archive_name']
if archive_name not in uploaded:
    raise RuntimeError('Missing source archive: ' + archive_name)
source_archive = Path(archive_name)
actual_archive_sha = hashlib.sha256(source_archive.read_bytes()).hexdigest()
assert actual_archive_sha == lock['package_sha256'], 'Source ZIP SHA mismatch'
run_id = __import__('datetime').datetime.now(__import__('datetime').timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
ROOT = Path('/content/ergt_runs') / EXPERIMENT_ID / REVISION / run_id / 'source'
ROOT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(source_archive) as archive:
    seen = set()
    total = 0
    for item in archive.infolist():
        name = item.filename
        parts = name.split('/')
        if not name or '\\' in name or ':' in name or name.startswith('/') or any(p in ('', '.', '..') for p in parts):
            raise RuntimeError('Unsafe ZIP path: ' + name)
        if name.casefold() in seen or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Duplicate or symlink ZIP entry: ' + name)
        seen.add(name.casefold())
        total += item.file_size
        if total > 2 * 1024**3:
            raise RuntimeError('Unexpectedly large source archive')
    manifest_bytes = archive.read('PACKAGE_MANIFEST.json')
    assert hashlib.sha256(manifest_bytes).hexdigest() == lock['package_manifest_sha256']
    package_manifest = json.loads(manifest_bytes)
    assert set(archive.namelist()) == set(package_manifest['files']) | {'PACKAGE_MANIFEST.json'}
    archive.extractall(ROOT)
for relative, expected in package_manifest['files'].items():
    assert hashlib.sha256((ROOT / relative).read_bytes()).hexdigest() == expected, relative
REVISION_DIR = ROOT / REVISION_RELATIVE
assert hashlib.sha256((REVISION_DIR / 'protocol.json').read_bytes()).hexdigest() == PROTOCOL_SHA256
assert hashlib.sha256((REVISION_DIR / 'experiment.ipynb').read_bytes()).hexdigest() == lock['notebook_sha256']
sys.path.insert(0, str(ROOT))
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
protocol = json.loads((REVISION_DIR / 'protocol.json').read_text(encoding='utf-8'))
print('Verified source:', EXPERIMENT_ID, REVISION, actual_archive_sha)


## ورودی‌ها و پیش‌نیازها

هر ورودی در `inputs` باید مسیر دقیق داخل پروژه، SHA-256 و محل دسترسی ثبت‌شده
داشته باشد. برای فایل‌های بزرگ از مسیر Drive استفاده کنید؛ در غیر این صورت
سلول برای هر فایل ثبت‌شده پنجرهٔ upload باز می‌کند. فایل‌ها تنها پس از تطبیق
hash کپی می‌شوند. هیچ checkpoint با نام مشابه جایگزین نسخهٔ ثبت‌شده نمی‌شود.

وابستگی تازه یا تغییر محیط باید در پروتکل نسخهٔ بعدی ثبت شود. این نوت‌بوک
به‌صورت پنهان package نصب نمی‌کند. محیط و خروجی خطای وابستگی نیز در bundle
اجرای ناموفق ثبت می‌شود.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
from research_tools.runner import safe_path, sha256

INPUT_ERRORS = []
for item in protocol.get('inputs', []):
    try:
        target = safe_path(ROOT, item['path'])
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_file() and sha256(target) == item['sha256']:
            continue
        locator = item.get('uri', '')
        candidate = Path(locator.removeprefix('file://')) if locator.startswith(('/content/drive/', 'file:///content/drive/')) else None
        if candidate is not None and candidate.is_file():
            assert sha256(candidate) == item['sha256'], 'Drive input SHA mismatch: ' + item['path']
            shutil.copy2(candidate, target)
        else:
            print('Upload input:', item['path'], 'Registered locator:', locator)
            input_upload = files.upload()
            matched = [name for name, data in input_upload.items() if hashlib.sha256(data).hexdigest() == item['sha256']]
            if len(matched) != 1:
                raise RuntimeError('Exactly one uploaded input must match its registered SHA')
            target.write_bytes(input_upload[matched[0]])
        assert sha256(target) == item['sha256']
    except Exception as exc:
        INPUT_ERRORS.append(item['path'] + ': ' + str(exc))
print('Input errors; runner will export preflight failure evidence:' if INPUT_ERRORS else 'All input locks passed', INPUT_ERRORS)


## اجرا و حفظ شواهد

Runner دقیقاً command ثبت‌شده را اجرا می‌کند. stdout/stderr، محیط، زمان، کد
خروج و تمام فایل‌های output_paths در ZIP نتیجه ثبت می‌شوند؛ شکست دستور هم
bundle دارد. پوشهٔ نتیجه روی Drive برای هر اجرا مستقل است و بازنویسی نمی‌شود.
در صورت قطع کامل runtime باید اجرای ناتمام ثبت و با run_id تازه تکرار شود.

موفقیت اجرایی (`returncode=0`) به معنی موفقیت علمی یا عبور فاز نیست.


In [ ]:
from research_tools.runner import run_protocol
DRIVE_RUN = Path('/content/drive/MyDrive/ERGT_Phi/research') / EXPERIMENT_ID / REVISION / run_id
DRIVE_RUN.mkdir(parents=True, exist_ok=False)
RESULT_BUNDLE = DRIVE_RUN / (run_id + '.zip')
run_manifest = run_protocol(ROOT, REVISION_DIR, RESULT_BUNDLE,
                            run_id=run_id, package_sha256=actual_archive_sha)
print(json.dumps(run_manifest, indent=2, ensure_ascii=False))
print('Preserved result bundle:', RESULT_BUNDLE)


## برگشت نتیجه و تصمیم بعدی

ZIP را دانلود و در `research/inbox/` ریپو قرار دهید. ایجنت باید آن را با دستور
workflow import دریافت کند، hash و ارتباط پروتکل/نوت‌بوک/سورس را بررسی کند،
و کنار همین revision گزارش تولید کند. سپس نتیجهٔ واقعی، کنترل‌ها، شکست‌ها و
گزینه‌های بعدی در review و دفتر تصمیم معماری ثبت شوند. هر پیشرفت باید commit
شود. اگر فایل نتیجه هنوز برنگشته، وضعیت «در انتظار نتیجه» است.

شناسهٔ پیگیری: `WF-E001/v001`. برای تکرار یک نسخه run_id تازه لازم است؛ برای
تغییر فرضیه، کد علمی یا معیار، revision تازه لازم است. این نوت‌بوک را برای
دورزدن معیارها در Colab ویرایش نکنید.


In [ ]:
files.download(str(RESULT_BUNDLE))
